## Checks for numeric variables - integer and float

### Planned checks:
 - range conformance
 - eventually apply stricter ranges
 - unit conformance

 ### Planned work
 - fetch ranges, formats and units from DD
 - generate rules
 - append rules to export_long

main()
│
├── run_general_rule_engine()
├── run_numeric_range_rules()
│   ├── get_numeric_dd()
│   ├── parse_numeric_ranges()
│   ├── validate_numeric_ranges()
│   └── make_violation()
└── combine all violations

In [33]:
import pandas as pd
from pathlib import Path
import operator
import re
DATA_DICTIONARY_PATH ="metadata/processed/data_dictionary.csv"
EXPORT_LONG_PATH = "data/processed/export_long_clean.csv"

OUTPUT_DIR = Path("results/numerics")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [34]:
dd=pd.read_csv(DATA_DICTIONARY_PATH, sep=";", dtype="object")
export_long=pd.read_csv(EXPORT_LONG_PATH, sep=";", dtype="object")

In [35]:
VALID_NUMERIC_TYPES = ["INTEGER", "FLOAT"]
OPS = {
    # maps operators in data_format (ranges) to functions of the operator module
    "<": operator.lt,
    "<=": operator.le,
    ">": operator.gt,
    ">=": operator.ge,
}


def get_numeric_types(df):
    # Return data dictionary rows for numeric variables
    df = df.copy()
    df["data_type"] = df["data_type"].str.upper().str.strip()

    cols = [
    "Form",
    "form_type",
    "record_designation",
    "dataelement_designation",
    "data_type",
    "data_format",
    "unit",
    "source_id"
]
    df = df[df["data_type"].isin(VALID_NUMERIC_TYPES)][cols]
    return df

get_numeric_types(dd)

,Form,form_type,record_designation,dataelement_designation,data_type,data_format,unit,source_id
11,ACLF evaluation,Basic,Liver failure,Total bilirubin,FLOAT,0<=x<=200,mg/dl,X_224_16_58
13,ACLF evaluation,Basic,Renal failure,Creatinine,FLOAT,0<=x<=200,mg/dl,X_224_17_60
16,ACLF evaluation,Basic,Coagulation failure,INR,FLOAT,0<=x<=100,NaN,X_224_18_65
20,ACLF evaluation,Basic,Circulatory failure,Systolic Blood Pressure,INTEGER,0<=x<=300,mmHg,X_224_19_67
21,ACLF evaluation,Basic,Circulatory failure,Diastolic Blood Pressure,INTEGER,0<=x<=300,mmHg,X_224_19_68
...,...,...,...,...,...,...,...,...
431,Scores,Longitudinal,CLIF-C AD score,Creatinine,FLOAT,0<=x,mg/dl,X_235_68_265
432,Scores,Longitudinal,CLIF-C AD score,INR,FLOAT,0<=x<=100,NaN,X_235_68_256
433,Scores,Longitudinal,CLIF-C AD score,Sodium,FLOAT,0<=x<=200,mmol/l,X_235_68_266
434,Scores,Longitudinal,NaN,CLIF-C AD score,INTEGER,0<=x<=200,NaN,X_235_0_91


In [36]:
def make_violation(df, rule_id, rule_type, expected, observed_col="source_value"):
    out = df.copy()

    if observed_col not in out.columns:
        raise ValueError(f"observed_col '{observed_col}' not found in dataframe")

    out["rule_id"] = rule_id
    out["rule_type"] = rule_type
    out["expected"] = expected
    out["observed"] = out[observed_col]

    return out

In [37]:
def parse_numeric_range(data_format):
    """
    Parses numeric range expressions from the data dictionary.

    Supported examples:
    - 0<X<100
    - 0<=X<=100
    - 0<X<=100
    - 0<=X<100
    - X<100
    - X<=100
    - X>0
    - X>=0
    """
    if data_format == "X":
        return []
    
    # standardize ranges
    range = str(data_format).replace(" ","").upper()

    conditions=[]

    # Case 1: 0<=X<=100 or 0<X<=100
    match= re.fullmatch( r"(-?\d+(?:\.\d+)?)(<=|<)X(<=|<)(-?\d+(?:\.\d+)?)",
        range
    )

    if match:
        lower_value, lower_operator, upper_operator, upper_value = match.groups()

        lower_value = float(lower_value)
        upper_value = float(upper_value)
        # data_format: lower < X
        # but we need condition from X perspective: X > lower
        if lower_operator == "<":
            conditions.append((">", lower_value))
        elif lower_operator == "<=":
            conditions.append((">=", lower_value))

        conditions.append((upper_operator, upper_value))

        return conditions
    
    # case 2: X < upper, X <= upper, X> lower, X >= lower
    match= re.fullmatch(r"X(<=|<|>=|>)(-?\d+(?:\.\d+)?)",
        range
    )
    if match:
        operator_symbol, value = match.groups()
        conditions.append((operator_symbol, float(value)))
        return conditions

    raise ValueError(f"Unsupported numeric range format: {data_format}")


"""""
Example: 
parse_numeric_range("0<X<20")
returns
[('>', 0.0), ('<', 20.0)]
"""""


'""\nExample: \nparse_numeric_range("0<X<20")\nreturns\n[(\'>\', 0.0), (\'<\', 20.0)]\n'

In [38]:
def check_numeric_range(df_export, df_dd):
    """
    Checks whether numeric source_values fall within the range defined    
    Parameters:
        df_export : the data export (contains source_id, source_value, data_type)
        df_dd     : the data dictionary (contains source_id, data_format, data_type)
    """
    
    # --- Step 1: get numeric rules from data dictionary ---
    # filters to INTEGER and FLOAT rows and keeps only relevant columns
    numeric_rules = get_numeric_types(df_dd)[["source_id", "data_format"]].copy()


 # --- Step 2: merge export with rules on source_id ---
    # this attaches data_type and data_format to each export row
    merged = df_export.merge(numeric_rules, on="source_id", how="inner")
    # inner join: only keep export rows that have a matching numeric rule
    # rows with non-numeric source_ids are automatically excluded

    # --- Step 4: skip rows where data_format is X (no constraint defined) ---
    has_range = ~merged["data_format"].str.strip().str.upper().eq("X")
    range_rows = merged[has_range].copy()

    # --- Step 5: convert source_value to numeric ---
    # replace commas with dots to accept both decimal separators
    # errors="coerce" turns unparseable values into NaN (caught by dtype check elsewhere)
    range_rows["source_value_numeric"] = pd.to_numeric(
        range_rows["source_value"].astype(str).str.strip().str.replace(",", ".", regex=False),
        errors="coerce"
    )

    # only validate rows where numeric conversion succeeded
    has_numeric = range_rows["source_value_numeric"].notna()
    range_rows = range_rows[has_numeric].copy()

    # --- Step 6: groupby data_format, parse once, apply to whole group ---
    failed_parts = []
    unparseable = []

    for data_format, group in range_rows.groupby("data_format", dropna=False):
        
        # parse range string into list of (operator, bound) tuples
        try:
            conditions = parse_numeric_range(data_format)
        except ValueError:
            # log unparseable formats — do not crash the pipeline
            unparseable.append(data_format)
            continue

        if not conditions:
            # empty conditions means no constraint (e.g. "X") — skip
            continue

        values = group["source_value_numeric"]
        violation_mask = pd.Series(False, index=group.index)

        # apply each condition — a row fails if ANY condition is violated
        for op_symbol, bound in conditions:
            op_func = OPS[op_symbol]
            violated = ~values.apply(lambda v: op_func(v, bound))
            violation_mask |= violated

        failed = group[violation_mask].copy()

        if not failed.empty:
            failed_parts.append(
                make_violation(
                    failed,
                    rule_id="NUMERIC_RANGE_VIOLATION",
                    rule_type="range_conformance",
                    expected=f"value within range {data_format}",
                    observed_col="source_value"
                )
            )

    # --- Step 7: log unparseable formats for auditability ---
    if unparseable:
        print(
            f"[check_numeric_range] WARNING: {len(unparseable)} unparseable format(s) "
            f"skipped — consider extending parse_numeric_range():\n  "
            + "\n  ".join(str(f) for f in unparseable)
        )

    # --- Step 8: combine all violations, drop internal helper columns ---
    if failed_parts:
        result = pd.concat(failed_parts, ignore_index=True)
        result = result.drop(columns=["source_value_numeric", "data_type_clean"], errors="ignore")
        return result

    return pd.DataFrame()

In [39]:
check_numeric_range(export_long,dd)

[check_numeric_range] WARNING: 1 unparseable format(s) skipped — consider extending parse_numeric_range():
  0<=x


,Unnamed: 0,PID,Episode,Episode_Date,IX,source_id,source_value,data_format,rule_id,rule_type,expected,observed
0,91089,69,2,02/03/2022,1,X_231_0_348,44444444444,0<=x<=300,NUMERIC_RANGE_VIOLATION,range_conformance,value within range 0<=x<=300,44444444444
